In [1]:
import requests
from bs4 import BeautifulSoup
import datetime

In [2]:
# Save data to a local text file
def save_to_file(data):
    try:
        with open('weather_data.txt', 'a') as f:
            f.write(f"{datetime.datetime.now()}: {data}\n")
        print("Data saved to text file.")
    except Exception as e:
        print(f"Failed to save data to file: {e}")

#Scrape Wunderground weather
def scrape_wunderground(city):
    url = f"https://www.wunderground.com/weather/se/{city.replace(' ', '-')}"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raises an exception for HTTP errors
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Extract temperature and weather condition
        temp = soup.find('span', class_='wu-value wu-value-to').get_text(strip=True)
        condition = soup.find('div', class_='condition-icon small-6 medium-12 columns').get_text(strip=True)
        return {'site': 'Wunderground', 'city': city, 'temperature': temp, 'condition': condition}
    except Exception as e:
        print(f"Error scraping Wunderground: {e}")
        return None

# Scrape TimeandDate weather
def scrape_timeanddate(city):
    url = f"https://www.timeanddate.com/weather/sweden/{city.replace(' ', '-')}"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raises an exception for HTTP errors
        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract the current temperature and condition
        temp_el = soup.find('div', class_='h2')
        condition_el = soup.find('p', class_='')

        if temp_el is None or condition_el is None:
            raise ValueError("Could not find the temperature or condition elements on the page.")

        temp = temp_el.text.replace(u'\xa0', u'')
        condition = condition_el.text

        return {'site': 'TimeandDate', 'city': city, 'temperature': temp, 'condition': condition}
    
    except Exception as e:
        print(f"Error scraping TimeandDate: {e}")
        return None

In [3]:
# Main function to scrape and save data
def main(city):
    # Scrape weather data from both websites
    timeanddate_data = scrape_timeanddate(city)
    wunderground_data = scrape_wunderground(city)

    # Handle None (errors) in scraped data
    data_to_save = []
    if timeanddate_data:
        data_to_save.append(timeanddate_data)
    if wunderground_data:
        data_to_save.append(wunderground_data)

    # Save data to a file
    if data_to_save:
        save_to_file(data_to_save)  # Save locally
    else:
        print("No data to save.")

if __name__ == "__main__":
    city_name = "linkoping"  # Example city
    main(city_name)

Data saved to text file.
